In [15]:
from pathlib import Path

import pandas as pd

In [16]:
df2_path = "../../data/dataset2/dataset2.csv"
df2_weather_path = "../../data/weather/dataset2_weather.csv"
df2_raw = pd.read_csv(df2_path)
weather_raw = pd.read_csv(df2_weather_path)

print('Dataset 2 raw shape:', df2_raw.shape)
print('Weather raw shape:', weather_raw.shape)

display(df2_raw.head())
display(weather_raw.head())

Dataset 2 raw shape: (149116, 11)
Weather raw shape: (13032, 25)


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


,location,city,borough,datetime,date,hour,temperature_c,apparent_temperature_c,relative_humidity,precipitation_mm,...,cloud_cover_pct,is_rainy,is_snowy,is_precipitation,weather_latitude,weather_longitude,timezone,utc_offset_seconds,weather_source,weather_spatial_level
0,Astoria,New York,Queens,2023-01-01 00:00:00,2023-01-01,0,10.9,9.8,99,1.2,...,100,True,False,True,40.7644,-73.9235,America/New_York,-14400,Open-Meteo Historical Weather API,area
1,Astoria,New York,Queens,2023-01-01 01:00:00,2023-01-01,1,10.6,9.8,99,1.2,...,100,True,False,True,40.7644,-73.9235,America/New_York,-14400,Open-Meteo Historical Weather API,area
2,Astoria,New York,Queens,2023-01-01 02:00:00,2023-01-01,2,10.5,8.8,98,0.2,...,61,True,False,True,40.7644,-73.9235,America/New_York,-14400,Open-Meteo Historical Weather API,area
3,Astoria,New York,Queens,2023-01-01 03:00:00,2023-01-01,3,10.6,8.3,96,0.0,...,7,False,False,False,40.7644,-73.9235,America/New_York,-14400,Open-Meteo Historical Weather API,area
4,Astoria,New York,Queens,2023-01-01 04:00:00,2023-01-01,4,9.9,7.5,94,0.0,...,1,False,False,False,40.7644,-73.9235,America/New_York,-14400,Open-Meteo Historical Weather API,area


In [17]:
df2 = df2_raw.copy()

df2['transaction_datetime'] = pd.to_datetime(
    df2['transaction_date'].astype(str) + ' ' + df2['transaction_time'].astype(str),
    errors='coerce',
)

df2['date'] = df2['transaction_datetime'].dt.normalize()
df2['hour'] = df2['transaction_datetime'].dt.hour
df2['location'] = df2['store_location']

df2['total_amount'] = df2['transaction_qty'] * df2['unit_price']

print('Invalid transaction_datetime:', df2['transaction_datetime'].isna().sum())
print('Dataset 2 date range:', df2['date'].min(), '->', df2['date'].max())
print('Dataset 2 locations:')
display(df2['location'].value_counts())

display(df2[['transaction_date', 'transaction_time', 'transaction_datetime', 'date', 'hour', 'location', 'total_amount']].head())

Invalid transaction_datetime: 0
Dataset 2 date range: 2023-01-01 00:00:00 -> 2023-06-30 00:00:00
Dataset 2 locations:


location
Hell's Kitchen     50735
Astoria            50599
Lower Manhattan    47782
Name: count, dtype: int64

,transaction_date,transaction_time,transaction_datetime,date,hour,location,total_amount
0,2023-01-01,07:06:11,2023-01-01 07:06:11,2023-01-01,7,Lower Manhattan,6.0
1,2023-01-01,07:08:56,2023-01-01 07:08:56,2023-01-01,7,Lower Manhattan,6.2
2,2023-01-01,07:14:04,2023-01-01 07:14:04,2023-01-01,7,Lower Manhattan,9.0
3,2023-01-01,07:20:24,2023-01-01 07:20:24,2023-01-01,7,Lower Manhattan,2.0
4,2023-01-01,07:22:41,2023-01-01 07:22:41,2023-01-01,7,Lower Manhattan,6.2


In [18]:
weather = weather_raw.copy()
weather['date'] = pd.to_datetime(weather['date'], errors='coerce').dt.normalize()
weather['hour'] = pd.to_numeric(weather['hour'], errors='coerce').astype('Int64')

weather_cols = [
    'date',
    'hour',
    'location',
    'city',
    'borough',
    'temperature_c',
    'apparent_temperature_c',
    'relative_humidity',
    'precipitation_mm',
    'rain_mm',
    'weather_code',
    'weather_code_description',
    'weather_condition',
    'wind_speed_10m_kmh',
    'cloud_cover_pct',
    'is_rainy',
    'is_snowy',
    'is_precipitation',
    'weather_latitude',
    'weather_longitude',
    'weather_source',
    'weather_spatial_level',
]

weather = weather[weather_cols].copy()

print('Weather duplicate join keys:', weather.duplicated(['date', 'hour', 'location']).sum())
print('Weather date range:', weather['date'].min(), '->', weather['date'].max())
print('Weather locations:')
display(weather['location'].value_counts())

display(weather.head())

Weather duplicate join keys: 0
Weather date range: 2023-01-01 00:00:00 -> 2023-06-30 00:00:00
Weather locations:


location
Astoria            4344
Hell's Kitchen     4344
Lower Manhattan    4344
Name: count, dtype: int64

,date,hour,location,city,borough,temperature_c,apparent_temperature_c,relative_humidity,precipitation_mm,rain_mm,...,weather_condition,wind_speed_10m_kmh,cloud_cover_pct,is_rainy,is_snowy,is_precipitation,weather_latitude,weather_longitude,weather_source,weather_spatial_level
0,2023-01-01,0,Astoria,New York,Queens,10.9,9.8,99,1.2,1.2,...,Rain,9.7,100,True,False,True,40.7644,-73.9235,Open-Meteo Historical Weather API,area
1,2023-01-01,1,Astoria,New York,Queens,10.6,9.8,99,1.2,1.2,...,Rain,6.7,100,True,False,True,40.7644,-73.9235,Open-Meteo Historical Weather API,area
2,2023-01-01,2,Astoria,New York,Queens,10.5,8.8,98,0.2,0.2,...,Rain,12.1,61,True,False,True,40.7644,-73.9235,Open-Meteo Historical Weather API,area
3,2023-01-01,3,Astoria,New York,Queens,10.6,8.3,96,0.0,0.0,...,Clear,15.5,7,False,False,False,40.7644,-73.9235,Open-Meteo Historical Weather API,area
4,2023-01-01,4,Astoria,New York,Queens,9.9,7.5,94,0.0,0.0,...,Clear,15.2,1,False,False,False,40.7644,-73.9235,Open-Meteo Historical Weather API,area


In [19]:
dataset2_locations = set(df2['location'].dropna().unique())
weather_locations = set(weather['location'].dropna().unique())

print('Locations in Dataset 2 but not in weather:', sorted(dataset2_locations - weather_locations))
print('Locations in weather but not in Dataset 2:', sorted(weather_locations - dataset2_locations))

df2_join_keys = df2[['date', 'hour', 'location']].drop_duplicates()
weather_join_keys = weather[['date', 'hour', 'location']].drop_duplicates()

key_check = df2_join_keys.merge(
    weather_join_keys.assign(has_weather_key=True),
    on=['date', 'hour', 'location'],
    how='left',
)

print('Dataset 2 unique join keys:', len(df2_join_keys))
print('Join keys without weather:', key_check['has_weather_key'].isna().sum())

display(key_check[key_check['has_weather_key'].isna()].head(20))

Locations in Dataset 2 but not in weather: []
Locations in weather but not in Dataset 2: []
Dataset 2 unique join keys: 6863
Join keys without weather: 0


,date,hour,location,has_weather_key


In [20]:
df2_with_weather = df2.merge(
    weather,
    on=['date', 'hour', 'location'],
    how='left',
    validate='many_to_one',
)

print('Dataset 2 before join:', df2.shape)
print('Dataset 2 after join:', df2_with_weather.shape)

display(df2_with_weather.head())

Dataset 2 before join: (149116, 16)
Dataset 2 after join: (149116, 35)


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,...,weather_condition,wind_speed_10m_kmh,cloud_cover_pct,is_rainy,is_snowy,is_precipitation,weather_latitude,weather_longitude,weather_source,weather_spatial_level
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,...,Cloudy,13.2,58,False,False,False,40.7075,-74.0113,Open-Meteo Historical Weather API,area
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,...,Cloudy,13.2,58,False,False,False,40.7075,-74.0113,Open-Meteo Historical Weather API,area
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,...,Cloudy,13.2,58,False,False,False,40.7075,-74.0113,Open-Meteo Historical Weather API,area
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,...,Cloudy,13.2,58,False,False,False,40.7075,-74.0113,Open-Meteo Historical Weather API,area
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,...,Cloudy,13.2,58,False,False,False,40.7075,-74.0113,Open-Meteo Historical Weather API,area


In [21]:
weather_check_cols = [
    'temperature_c',
    'apparent_temperature_c',
    'relative_humidity',
    'precipitation_mm',
    'rain_mm',
    'weather_code',
    'weather_condition',
    'wind_speed_10m_kmh',
    'cloud_cover_pct',
]

missing_weather = df2_with_weather[weather_check_cols].isna().sum().sort_values(ascending=False)
missing_weather_pct = (df2_with_weather[weather_check_cols].isna().mean() * 100).sort_values(ascending=False)

weather_missing_summary = pd.DataFrame({
    'missing_count': missing_weather,
    'missing_pct': missing_weather_pct,
})

print('Missing weather after join:')
display(weather_missing_summary)

print('Weather condition distribution:')
display(df2_with_weather['weather_condition'].value_counts(dropna=False))

print('Temperature summary by location:')
display(df2_with_weather.groupby('location')['temperature_c'].describe())

Missing weather after join:


,missing_count,missing_pct
temperature_c,0,0.0
apparent_temperature_c,0,0.0
relative_humidity,0,0.0
precipitation_mm,0,0.0
rain_mm,0,0.0
weather_code,0,0.0
weather_condition,0,0.0
wind_speed_10m_kmh,0,0.0
cloud_cover_pct,0,0.0


Weather condition distribution:


weather_condition
Cloudy    82041
Clear     46480
Rain      18448
Snow       2147
Name: count, dtype: int64

Temperature summary by location:


,count,mean,std,min,25%,50%,75%,max
location,,,,,,,,
Astoria,50599.0,13.448568,7.994995,-11.4,7.1,13.8,20.0,31.0
Hell's Kitchen,50735.0,12.873078,8.112640,-15.0,6.2,13.1,19.5,31.2
Lower Manhattan,47782.0,12.736539,8.106920,-16.4,6.2,13.1,19.2,31.1


In [22]:
OUTPUT_PATH = Path("../../data/dataset2/dataset2_with_weather.csv")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df2_with_weather.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("Saved Dataset 2 with weather to:", OUTPUT_PATH)
print("Final shape:", df2_with_weather.shape)

Saved Dataset 2 with weather to: ../../data/dataset2/dataset2_with_weather.csv
Final shape: (149116, 35)
